## **1. Data Ingestion**


In [2]:
from langchain_core.documents import Document

In [3]:
#  examaple 
doc = Document(
    page_content="I am using this as to create RAG Piplene and this is the main text content.",
    metadata ={
        "source": "demo.txt",
        "pages":1,
        "author":"Harshit Singh",
        "date_created": "07-03-2026"
    }
)
doc

Document(metadata={'source': 'demo.txt', 'pages': 1, 'author': 'Harshit Singh', 'date_created': '07-03-2026'}, page_content='I am using this as to create RAG Piplene and this is the main text content.')

In [4]:
# create a txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath, content in sample_texts.items():
    with open(filepath, "w") as f:
        f.write(content)
        
print("Sample text files created successfully.")        

Sample text files created successfully.


In [6]:
# Textloader using langchain
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [7]:
# Directory Loader

from langchain_community.document_loaders import DirectoryLoader

# load all text files from the directory

dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt", # Pattern to match all text files in the directory and subdirectories
    loader_cls=TextLoader, # Specify the loader class to use for each file
    loader_kwargs={"encoding":"utf-8"}, 
    show_progress=False
)

documets = dir_loader.load()
print(documets)

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '), Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popul

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

# PDF Loader using
dir_loader = DirectoryLoader(
    "../data/pdf", 
    glob="**/*.pdf", # Pattern to match all PDF files in the directory and subdirectories
    loader_cls=PyMuPDFLoader, # Specify the loader class to use for each file
    show_progress=False
)

pdf_documents = dir_loader.load()
print(pdf_documents)

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2025-11-03T16:43:02+05:30', 'source': '..\\data\\pdf\\Attention is all you need.pdf', 'file_path': '..\\data\\pdf\\Attention is all you need.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '1706.03762', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-11-03T16:43:02+05:30', 'trapped': '', 'modDate': "D:20251103164302+05'30'", 'creationDate': "D:20251103164302+05'30'", 'page': 0}, page_content=''), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2025-11-03T16:43:02+05:30', 'source': '..\\data\\pdf\\Attention is all you need.pdf', 'file_path': '..\\data\\pdf\\Attention is all you need.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '1706.03762', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-11-03T16:43:02+05:30', 'trapped': '', 'modDate': "D:20251103164302+05'30'", 'creationDate': "D:20251103164302+05'30'", 'page':

In [9]:
type(pdf_documents[0])

langchain_core.documents.base.Document

## **2. Embedding And VectorDB**

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingStore:
    """Handles doc embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
             model_name: HuggingFace model name for the sentence embedding
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise   
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """"""    